# Model Comparison — All Architectures

Compares all 7 trained models on:
- ROC curves (micro + macro AUC) on one graph
- Accuracy, Precision, Recall, F1, Specificity
- Parameter count & memory
- Training time
- Per-metric bar charts

> **Run this notebook AFTER all 7 notebooks have been executed.**


In [ ]:
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from itertools import cycle
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
warnings.filterwarnings('ignore')

NOTEBOOK_IDS = ['NB1_Wheat_Hybrid', 'NB2_Cotton_Hybrid', 'NB3_PlantVillage_Hybrid', 'NB4_Combined_StandardInception', 'NB5_Combined_ResNet', 'NB6_Combined_InceptionAdd', 'NB7_Combined_SkipConcat']
NB_LABELS    = {'NB1_Wheat_Hybrid': 'Wheat-Hybrid', 'NB2_Cotton_Hybrid': 'Cotton-Hybrid', 'NB3_PlantVillage_Hybrid': 'PV-Hybrid', 'NB4_Combined_StandardInception': 'Standard Inception', 'NB5_Combined_ResNet': 'ResNet-style', 'NB6_Combined_InceptionAdd': 'Inception+Add', 'NB7_Combined_SkipConcat': 'Skip+Concat'}
NB_COLORS    = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653', '#A8DADC']
BASE_ROOT    = os.path.join(os.path.expanduser('~'), 'plant_project')

print('Comparison notebook ready.')
print(f'Looking for models in: {BASE_ROOT}')


In [ ]:
# Load all metrics CSVs
all_metrics = []
missing = []
for nb_id in NOTEBOOK_IDS:
    csv_path = os.path.join(BASE_ROOT, nb_id, 'logs', 'metrics_summary.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['label'] = NB_LABELS[nb_id]
        all_metrics.append(df)
        print(f'  Loaded: {nb_id}')
    else:
        print(f'  MISSING: {csv_path}')
        missing.append(nb_id)

if missing:
    print(f'\nWARNING: {len(missing)} notebooks not yet run.')
    print('Run all 7 notebooks first, then re-run this comparison.')

if all_metrics:
    metrics_df = pd.concat(all_metrics, ignore_index=True)
    print(f'\nLoaded {len(metrics_df)} model results.')
    display_cols = ['label','accuracy','precision','recall','f1','specificity',
                    'params','train_time_min','ram_mb','num_classes']
    print(metrics_df[[c for c in display_cols if c in metrics_df.columns]].to_string(index=False))


In [ ]:
# Load all training histories
all_hist = []
for nb_id in NOTEBOOK_IDS:
    csv_path = os.path.join(BASE_ROOT, nb_id, 'logs', 'training_history.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['label'] = NB_LABELS[nb_id]
        all_hist.append(df)

if all_hist:
    hist_df = pd.concat(all_hist, ignore_index=True)
    print(f'Training histories loaded: {len(all_hist)} notebooks')
else:
    hist_df = pd.DataFrame()
    print('No training histories found.')


In [ ]:
# ── Combined ROC curves (macro-avg per model) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

loaded_roc = []
for nb_id, color in zip(NOTEBOOK_IDS, NB_COLORS):
    logs_dir = os.path.join(BASE_ROOT, nb_id, 'logs')
    prob_path = os.path.join(logs_dir, 'y_prob.npy')
    true_path = os.path.join(logs_dir, 'y_true.npy')
    map_path  = os.path.join(BASE_ROOT, nb_id, 'class_mapping.json')
    if not (os.path.exists(prob_path) and os.path.exists(true_path)):
        print(f'  Skipping {nb_id} — predictions not found'); continue
    y_prob = np.load(prob_path)
    y_true = np.load(true_path)
    n_cls  = y_prob.shape[1]
    with open(map_path) as f: mapping = json.load(f)
    y_bin  = label_binarize(y_true, classes=list(range(n_cls)))

    # Micro AUC
    fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
    micro_auc = auc(fpr_m, tpr_m)

    # Macro AUC
    all_fpr = np.unique(np.concatenate([
        roc_curve(y_bin[:,i], y_prob[:,i])[0] for i in range(n_cls)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_cls):
        fpr_i,tpr_i,_ = roc_curve(y_bin[:,i], y_prob[:,i])
        mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
    mean_tpr /= n_cls
    macro_auc = auc(all_fpr, mean_tpr)

    label = NB_LABELS[nb_id]
    axes[0].plot(fpr_m, tpr_m, color=color, lw=2,
                 label=f'{label}  (AUC={micro_auc:.4f})')
    axes[1].plot(all_fpr, mean_tpr, color=color, lw=2, ls='--',
                 label=f'{label}  (AUC={macro_auc:.4f})')
    loaded_roc.append({'label':label,'micro_auc':micro_auc,'macro_auc':macro_auc})
    print(f'  {label:<25} micro={micro_auc:.4f}  macro={macro_auc:.4f}')

for ax, title in [(axes[0],'Micro-average ROC'), (axes[1],'Macro-average ROC')]:
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('False Positive Rate',fontsize=11)
    ax.set_ylabel('True Positive Rate',fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.25)

plt.suptitle('All Models — ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_ROOT,'comparison_roc.png'),dpi=150,bbox_inches='tight')
plt.show()


In [ ]:
# ── Bar chart comparison: all metrics ─────────────────────────────────────
if not all_metrics: print('No metrics loaded.') 
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metric_axes = [
        ('accuracy',     axes[0,0], 'Accuracy'),
        ('f1',           axes[0,1], 'F1-Score (weighted)'),
        ('precision',    axes[0,2], 'Precision (weighted)'),
        ('recall',       axes[1,0], 'Recall / Sensitivity'),
        ('specificity',  axes[1,1], 'Specificity'),
        ('best_val_acc', axes[1,2], 'Best Val Accuracy'),
    ]
    labels = [NB_LABELS[n] for n in NOTEBOOK_IDS if n in metrics_df['notebook'].values]
    for metric, ax, title in metric_axes:
        if metric not in metrics_df.columns: ax.set_visible(False); continue
        vals = [metrics_df[metrics_df['notebook']==n][metric].values[0]
                if n in metrics_df['notebook'].values else 0
                for n in NOTEBOOK_IDS]
        bars = ax.bar(range(len(vals)), vals, color=NB_COLORS[:len(vals)], edgecolor='white')
        for bar,v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='500')
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels([NB_LABELS[n] for n in NOTEBOOK_IDS],
                           rotation=30, ha='right', fontsize=8)
        ax.set_ylim(0,1.12); ax.set_title(title,fontweight='bold')
        ax.grid(True,axis='y',alpha=0.3)
    plt.suptitle('All Models — Performance Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_ROOT,'comparison_metrics.png'),dpi=150,bbox_inches='tight')
    plt.show()


In [ ]:
# ── Parameter count + memory scatter ──────────────────────────────────────
if not all_metrics: print('No metrics.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i,(n,color) in enumerate(zip(NOTEBOOK_IDS, NB_COLORS)):
        row = metrics_df[metrics_df['notebook']==n]
        if row.empty: continue
        label = NB_LABELS[n]
        params = row['params'].values[0]/1e6
        acc    = row['accuracy'].values[0]
        f1     = row['f1'].values[0]
        ram    = row.get('ram_mb',pd.Series([0])).values[0]
        ttime  = row.get('train_time_min',pd.Series([0])).values[0]
        axes[0].scatter(params, acc, color=color, s=120, zorder=5, label=label)
        axes[0].annotate(label, (params,acc), textcoords='offset points',
                         xytext=(5,5), fontsize=7)
        axes[1].scatter(ttime, f1, color=color, s=120, zorder=5, label=label)
        axes[1].annotate(label, (ttime,f1), textcoords='offset points',
                         xytext=(5,5), fontsize=7)
        axes[2].bar(i, params, color=color)

    axes[0].set_xlabel('Parameters (M)'); axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Accuracy vs. Model Size',fontweight='bold'); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel('Train Time (min)'); axes[1].set_ylabel('F1-Score')
    axes[1].set_title('F1 vs. Training Time',fontweight='bold'); axes[1].grid(alpha=0.3)
    axes[2].set_xticks(range(len(NOTEBOOK_IDS)))
    axes[2].set_xticklabels([NB_LABELS[n] for n in NOTEBOOK_IDS],rotation=30,ha='right',fontsize=8)
    axes[2].set_ylabel('Parameters (M)'); axes[2].set_title('Model Size',fontweight='bold')
    axes[2].grid(True,axis='y',alpha=0.3)
    for bar,color in zip(axes[2].patches,NB_COLORS):
        bar.set_color(color)

    plt.suptitle('Model Efficiency Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_ROOT,'comparison_efficiency.png'),dpi=150,bbox_inches='tight')
    plt.show()


In [ ]:
# ── Learning curves overlay ────────────────────────────────────────────────
if hist_df.empty: print('No histories loaded.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for nb_id, color in zip(NOTEBOOK_IDS, NB_COLORS):
        label = NB_LABELS[nb_id]
        sub = hist_df[hist_df['label']==label]
        if sub.empty: continue
        ep = sub['epoch'] if 'epoch' in sub else range(1,len(sub)+1)
        if 'val_accuracy' in sub.columns:
            axes[0].plot(ep, sub['val_accuracy'], color=color, lw=2, label=label)
        if 'val_loss' in sub.columns:
            axes[1].plot(ep, sub['val_loss'],     color=color, lw=2, label=label)
    for ax,title,ylabel in [(axes[0],'Val Accuracy','Accuracy'),(axes[1],'Val Loss','Loss')]:
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(title,fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.suptitle('Validation Curves — All Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_ROOT,'comparison_val_curves.png'),dpi=150,bbox_inches='tight')
    plt.show()


In [ ]:
# ── Combined summary table ─────────────────────────────────────────────────
if not all_metrics: print('No data.')
else:
    summary_cols = ['label','accuracy','precision','recall','f1','specificity',
                    'params','train_time_min','num_classes']
    summary = metrics_df[[c for c in summary_cols if c in metrics_df.columns]].copy()
    summary = summary.rename(columns={'label':'Model','accuracy':'Acc','precision':'Prec',
                                       'recall':'Recall','f1':'F1','specificity':'Spec',
                                       'params':'Params','train_time_min':'Train(min)',
                                       'num_classes':'Classes'})
    for col in ['Acc','Prec','Recall','F1','Spec']:
        if col in summary.columns:
            summary[col] = summary[col].map(lambda x: f'{x:.4f}')
    if 'Params' in summary.columns:
        summary['Params'] = summary['Params'].map(lambda x: f'{int(x):,}')
    if 'Train(min)' in summary.columns:
        summary['Train(min)'] = summary['Train(min)'].map(lambda x: f'{x:.1f}')
    print('\n' + '='*90)
    print('COMPLETE MODEL COMPARISON')
    print('='*90)
    print(summary.to_string(index=False))
    print('='*90)
    summary.to_csv(os.path.join(BASE_ROOT,'comparison_summary.csv'), index=False)
    print(f'\nSummary CSV saved: {BASE_ROOT}/comparison_summary.csv')

roc_summary = pd.DataFrame(loaded_roc) if 'loaded_roc' in dir() and loaded_roc else pd.DataFrame()
if not roc_summary.empty:
    print('\nROC AUC Summary:')
    print(roc_summary.to_string(index=False))
    roc_summary.to_csv(os.path.join(BASE_ROOT,'comparison_roc_auc.csv'), index=False)
